# Python Calculator Complete Project

### **Steps**

- Setup & Imports
- Custom Exception
- Logging Setup
- Calculator Class
- Demo Run - Noraml Cases & Error Cases
- Test
- Move from notebook to a real project (.py files)

![alt text](../images/aaii_logo.png)

## Setup & Imports

An **import** makes code from Python's standard library available in this notebook.

- `logging` writes useful messages about what the program is doing.
- `math` provides reliable mathematical functions such as `sqrt()`

In [1]:
import logging
import math
import pytest

from pathlib import Path

In [2]:
# !pip install pytest # Uncomment this line if pytest is not installed in your environment

## Custom Exceptions

An exception is an error that interrupts normal code. Python already has exceptions such as `ZeroDivisionError`, but a calculator benefits from its own names.

`CalculatorBaseError` is the parent class. Every other calculator error inherits from it. This lets us either catch one specific error or catch **any** calculator error.

The triple-quoted text under each class is a **docstring**: it documents the class for readers and tools, but it is not the error message shown to a user.


```text
Exception
└── CalculatorBaseError
    ├── ZeroDivisionError
```

For example, `ZeroDivisionError("Cannot divide by zero.")` creates a specific error with a helpful message.

In [3]:
class CalculatorBaseError(Exception): # Base class is inherited from Exception class
    """Base class for exceptions in this module."""
    pass

class ZeroDivisionError(CalculatorBaseError): # ZeroDivisionError class is inherited from CalculatorBaseError class
    """Exception raised when attempting to divide by zero."""
    pass

## Logging Setup/Config


Logging is not the same as `print()`:

- `print()` is mainly for normal output a user should see.
- logging records events for developers: information, warnings, and errors.

`basicConfig()` configures logging once. The format uses placeholders:

- `%(asctime)s`: date and time;
- `%(name)s`: which part of the program created the message;
- `%(levelname)s`: `INFO`, `WARNING`, or `ERROR`;
- `%(message)s`: the message itself.

In [4]:
def configure_log():
    
    log_file = 'calculator.log' # log_file = Path("logs/calculator.log")
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        # 2024-06-15 12:00:00,000 - root - INFO - This is an info message.
        handlers=[
            logging.FileHandler(log_file), # file -> send log messages to a file calculator.log
            logging.StreamHandler() # stream -> send log messages to the console/terminal
        ],
        force=True # This will override any existing logging configuration
    )

In [5]:
configure_log() # Call the configure_log function to set up logging configuration

logger = logging.getLogger("calculator") # Create a logger named "calculator" or get the existing logger with that name
logger.info("Calculator module initialized.") # Log an info message indicating the calculator module has been initialized

2026-09-23 18:21:33,067 - calculator - INFO - Calculator module initialized.


## Calculator Class Setup


A **class** is a blueprint. `Calculator` describes what a calculator can do. Each function inside it is called a **method**.

Every method starts with `self`. When we call `calculator.add(2, 3)`, Python automatically supplies the object as `self`; we only provide `2` and `3`.

The pattern in a successful operation is:

1. Calculate the answer.
2. Write an `INFO` log message.
3. Return the answer with `return`.

The pattern for invalid input is:

1. Check the input with `if`.
2. Write an `ERROR` log message.
3. `raise` a specific custom exception.

In [6]:
class Calculator:
    """ Provides basic arithmetic operations.
        Operations include addition, subtraction, multiplication, division, power, modulus, and square root.
    """
    # we can use this dictionary to map operation names to method names
    OPERATIONS = { 
        "add": "add",
        "subtract": "subtract",
        "multiply": "multiply",
        "divide": "divide",
        "power": "power",
        "modulus": "modulus",
        "square_root": "square_root"
    }
    
    def add(self, a: float, b: float)-> float:
        """Returns the sum of a and b."""
        result = a + b
        logger.info(f"Adding {a} and {b}: Result = {result}")
        return result 
    
    def subtract(self, a: float, b: float)-> float:
        """Returns the difference of a and b."""
        result = a - b
        logger.info(f"Subtracting {b} from {a}: Result = {result}")
        return result
    
    def multiply(self, a: float, b: float)-> float:
        """Returns the product of a and b."""
        result = a * b
        logger.info(f"Multiplying {a} and {b}: Result = {result}")
        return result
    
    def divide(self, a: float, b: float)-> float:
        """Returns the quotient of a and b. Raises ZeroDivisionError if b is zero."""
        if b == 0:
            logger.error("Attempted to divide by zero.")
            raise ZeroDivisionError("Cannot divide by zero.")
            
        result = a / b
        logger.info(f"Dividing {a} by {b}: Result = {result}")
        return result
    
    def power(self, a: float, b: float)-> float:
        """Returns a raised to the power of b."""
        result = a ** b
        logger.info(f"Raising {a} to the power of {b}: Result = {result}")
        return result
    
    def modulus(self, a: float, b: float)-> float:
        """Returns the modulus of a and b."""
        if b == 0:
            logger.error("Attempted to compute modulus with divisor zero.")
            raise ZeroDivisionError("Cannot compute modulus with divisor zero.")
        result = a % b
        logger.info(f"Computing modulus of {a} and {b}: Result = {result}")
        return result
    
    def square_root(self, a: float)-> float:
        """Returns the square root of a. Raises CalculatorBaseError if a is negative."""
        if a < 0:
            logger.error("Attempted to compute square root of a negative number.")
            raise CalculatorBaseError("Cannot compute square root of a negative number.")
        
        result = math.sqrt(a)
        logger.info(f"Computing square root of {a}: Result = {result}")
        return result
    
    
    def operations(self, operation: str, *args) -> float:
        """Performs the specified operation with the given arguments."""
        method_name = self.OPERATIONS.get(operation)
        
        if method_name is None:
            logger.error(f"Invalid operation '{operation}' requested.") # ("Invalid operation %s", operation)
            raise CalculatorBaseError(f"Invalid operation '{operation}'. Supported operations are: {list(self.OPERATIONS.keys())}")
        
        # method name is "add" => self.add(*args)
        method =getattr(self, method_name) # self.add
        result = method(*args)
        logger.info(f"Performed operation '{operation}' with arguments {args}: Result = {result}")
        return result
        

`NOTE:` Why `%s` is used in log messages

This:

```python
logger.info("Added %s and %s: %s", a, b, result)
```

Asks logging to insert the values only when that log level is enabled. This is the common Python logging style. It is better than building an f-string for every log message.


## Demo Run

In [7]:
calculator = Calculator()

print(calculator.add(5, 3))  # Output: 8
print(calculator.subtract(10, 4))  # Output: 6
print(calculator.multiply(2, 3))   # Output: 6
print(calculator.divide(10, 2))    # Output: 5.0
print(calculator.power(2, 3))  # Output: 8
print(calculator.modulus(10, 3))  # Output: 1
print(calculator.square_root(16))  # Output: 4
print(calculator.operations("add", 5, 3))  # Output: 8

2026-09-23 18:21:33,161 - calculator - INFO - Adding 5 and 3: Result = 8
2026-09-23 18:21:33,165 - calculator - INFO - Subtracting 4 from 10: Result = 6
2026-09-23 18:21:33,167 - calculator - INFO - Multiplying 2 and 3: Result = 6
2026-09-23 18:21:33,170 - calculator - INFO - Dividing 10 by 2: Result = 5.0
2026-09-23 18:21:33,172 - calculator - INFO - Raising 2 to the power of 3: Result = 8
2026-09-23 18:21:33,175 - calculator - INFO - Computing modulus of 10 and 3: Result = 1
2026-09-23 18:21:33,178 - calculator - INFO - Computing square root of 16: Result = 4.0
2026-09-23 18:21:33,178 - calculator - INFO - Adding 5 and 3: Result = 8
2026-09-23 18:21:33,178 - calculator - INFO - Performed operation 'add' with arguments (5, 3): Result = 8


8
6
6
5.0
8
1
4.0
8


The calculator method raises an error because it cannot safely continue. The code that calls it decides what to do next.

- `try` contains code that might raise an exception.
- `except CalculatorBaseError` catches any custom calculator error.
- `as error` stores the error object so we can display its message.

Using the parent `CalculatorBaseError` means one `except` can handle all errors.

In [8]:
try:
    print(calculator.divide(10, 0))  # This will raise ZeroDivisionError
except ZeroDivisionError as e:
    print(e)

2026-09-23 18:21:33,210 - calculator - ERROR - Attempted to divide by zero.


Cannot divide by zero.


In [9]:
try:
    print(calculator.modulus(10, 0))  # This will raise ZeroDivisionError
except ZeroDivisionError as e:
    print(e)

2026-09-23 18:21:33,250 - calculator - ERROR - Attempted to compute modulus with divisor zero.


Cannot compute modulus with divisor zero.


In [10]:
try:
    print(calculator.square_root(-4))  # This will raise CalculatorBaseError
except CalculatorBaseError as e:
    print(e)

2026-09-23 18:21:33,273 - calculator - ERROR - Attempted to compute square root of a negative number.


Cannot compute square root of a negative number.


`operations()` is useful when the operation comes from text, for example from a command line, web form, or API.

1. The `OPERATIONS` dictionary finds the method name.
2. `getattr()` retrieves that method from the calculator object.
3. The method is called with the supplied numbers.

In [11]:
try:
    print(calculator.operations("subtractss", 5, 3))  # Output: 2
except CalculatorBaseError as e:
    print(f"Error: {e}")    

2026-09-23 18:21:33,297 - calculator - ERROR - Invalid operation 'subtractss' requested.


Error: Invalid operation 'subtractss'. Supported operations are: ['add', 'subtract', 'multiply', 'divide', 'power', 'modulus', 'square_root']


## Test

**Automated tests with `pytest`**

Tests check that code still works after changes.

`pytest.raises(...)` passes only when the expected error is raised. It is clearer and shorter than writing `try` / `except` manually in every test.

In a normal project, save these functions in `tests/test_calculator.py` and run `python -m pytest` in a terminal.

In [12]:
def test_basic_operations():
    assert calculator.add(1, 2) == 3
    assert calculator.subtract(5, 3) == 2
    assert calculator.multiply(4, 2) == 8
    assert calculator.divide(10, 2) == 5.0
    assert calculator.power(2, 3) == 8
    assert calculator.modulus(10, 3) == 1
    assert calculator.square_root(16) == 4
    assert calculator.operations("add", 5, 3) == 8
    
    
def test_divide_by_zero():
    with pytest.raises(ZeroDivisionError):
        calculator.divide(10, 0)
        
def test_modulus_by_zero():
    with pytest.raises(ZeroDivisionError):
        calculator.modulus(10, 0)
        
def test_square_root_of_negative():
    with pytest.raises(CalculatorBaseError):
        calculator.square_root(-4)
        
        
def test_invalid_operation():
    with pytest.raises(CalculatorBaseError):
        calculator.operations("invalid_op", 5, 3)

The following cell runs each test once. In a real project, prefer the `pytest` command because it discovers and reports tests automatically.


In [13]:
test_cases = [
    test_basic_operations,
    test_divide_by_zero,    
    test_modulus_by_zero,
    test_square_root_of_negative,
    test_invalid_operation
]


for test in test_cases:
    test()
    print(f"{test.__name__} passed.")
        

2026-09-23 18:21:33,340 - calculator - INFO - Adding 1 and 2: Result = 3
2026-09-23 18:21:33,342 - calculator - INFO - Subtracting 3 from 5: Result = 2
2026-09-23 18:21:33,344 - calculator - INFO - Multiplying 4 and 2: Result = 8
2026-09-23 18:21:33,344 - calculator - INFO - Dividing 10 by 2: Result = 5.0
2026-09-23 18:21:33,344 - calculator - INFO - Raising 2 to the power of 3: Result = 8
2026-09-23 18:21:33,347 - calculator - INFO - Computing modulus of 10 and 3: Result = 1
2026-09-23 18:21:33,349 - calculator - INFO - Computing square root of 16: Result = 4.0
2026-09-23 18:21:33,349 - calculator - INFO - Adding 5 and 3: Result = 8
2026-09-23 18:21:33,350 - calculator - INFO - Performed operation 'add' with arguments (5, 3): Result = 8
2026-09-23 18:21:33,353 - calculator - ERROR - Attempted to divide by zero.
2026-09-23 18:21:33,355 - calculator - ERROR - Attempted to compute modulus with divisor zero.
2026-09-23 18:21:33,355 - calculator - ERROR - Attempted to compute square root o

test_basic_operations passed.
test_divide_by_zero passed.
test_modulus_by_zero passed.
test_square_root_of_negative passed.
test_invalid_operation passed.


## Move from one notebook to a real project

When a project grows, split it by responsibility:

```text
CalSimp-python/
├── src/
│   ├── calculator.py       # Calculator class and operations
│   ├── exceptions.py       # Custom error classes
│   ├── logging_config.py   # Logging setup, called once at startup
│   ├── main.py             # Small demonstration program
├── tests/test_calculator.py
├── logs/calculator.log
└── README.md
```

In a multi-file project, `logging_config.py` configures logging once. Each other Python file uses:

```python
import logging
logger = logging.getLogger(__name__)
```

This gives useful names such as `calculator_app.calculator` in the logs and avoids importing one shared logger from a configuration file.